# Descarga S3 — dynamo_entities_data

Descarga recursiva del prefijo `Productos/Data Quality/dynamo_entities_data/` del bucket `com.edmachina.data-science-eu` (eu-central-1), replicando la estructura de subcarpetas en local.

Usa las credenciales ya configuradas (perfil por defecto / VPN).

In [ ]:
import boto3
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

BUCKET = "com.edmachina.data-science-eu"
PREFIX = "Productos/Data Quality/dynamo_entities_data/"
REGION = "eu-central-1"
DESTINO = Path("dynamo_entities_data")  # carpeta local de salida
SOBREESCRIBIR = False  # True para volver a bajar archivos ya existentes

# credenciales desde el .env del repo (no se copian acá para no versionarlas)
ENV_FILE = Path(r"D:\EdMachina\01. Repositorios\dagster_data_pipes\envs\local.env")
assert load_dotenv(ENV_FILE), f"No se pudo leer {ENV_FILE}"

s3 = boto3.client("s3", region_name=REGION)
print("Conectado como:", boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"])

In [ ]:
print(f"Listando objetos en s3://{BUCKET}/{PREFIX} ...")

objetos = [
    o
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX)
    for o in page.get("Contents", [])
    if not o["Key"].endswith("/")  # ignora los marcadores de carpeta
]

total_bytes = sum(o["Size"] for o in objetos)
print(f"{len(objetos)} archivos | {total_bytes / 1024**2:,.1f} MB")

In [ ]:
descargados = omitidos = 0

# barra única, fija: se actualiza in-place con bytes transferidos + ETA
with tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024, desc="Descargando") as barra:
    for obj in objetos:
        destino = DESTINO / Path(obj["Key"][len(PREFIX):])

        if not SOBREESCRIBIR and destino.exists() and destino.stat().st_size == obj["Size"]:
            omitidos += 1
            barra.update(obj["Size"])
            continue

        destino.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, obj["Key"], str(destino), Callback=barra.update)
        descargados += 1

print(f"\nListo: {descargados} descargados, {omitidos} ya existían -> {DESTINO.resolve()}")